In [ ]:
%pip install -q ipylab

# Autostart

Auto start-up is enabled via a the `pluggy` framework and works by loading `ipylab` in a kernel named `Ipylab backend` that automatically starts when `ipylab` is activated. Plugins are found using the entry point: "ipylab-python-backend". Autostart simply calls `Python code`. 

## Entry points

Add the following in your `pyproject.toml`

``` toml
[project.entry-points.ipylab-python-backend]
my_plugins = "my_module.autostart"
```

## Example lauching a small app

In [ ]:
# @my_module.autostart.py

import asyncio

import ipylab

app = ipylab.JupyterFrontEnd()


def create_app():
    # Ensure this function provides all the imports.
    import ipywidgets as ipw

    import ipylab

    ma = ipylab.MainArea(name="My demo app")
    console_button = ipw.Button(description="Toggle console")
    console_button.on_click(
        lambda b: ma.load_console() if not ma.console_loaded else ma.unload_console()
    )
    ma.content.children = [
        ipw.HTML(f"<h3>My simple app</h3> Welcome to my app.<br> kernel id: {ma.kernelId}"),
        console_button,
    ]
    ma.load()
    print("Finshed creating my app")

    # Retun ma so it doesn't accidentally get garbage collected.
    return ma


async def load_app_async():
    session = await app.newNotebook(path="my app")
    kernelId = session["kernel"]["id"]
    await app.injectCode(kernelId, create_app)
    print("Done")


class MyPlugins:
    @ipylab.hookspecs.hookimpl()
    def run_once_at_startup(self):
        asyncio.create_task(load_app_async())


MyPlugins()

### Lauch app
Simulate code launch in the as it happens in `Ipylab backend`

In [ ]:
# Register plugin (normally via the entry point `ipylab-python-backend`)
ipylab.hookspecs.pm.register(MyPlugins())

# Called when Ipylab is activated and Ipylab backend launches
app._init_python_backend()